**Rigorous Distillation Column: Benzene-Toluene**

In this tutorial we model a rigorous, equilibrium stage distillation column for benzene-toluene with the IDAES `TrayColumn` model, and validate it stage by stage against DWSIM's Distillation Column (Peng-Robinson property package). The column is the one sized with the shortcut method in the Shortcut Column tutorial.

On every stage the model solves the **MESH** equations: **M**aterial balances, phase **E**quilibrium (Peng-Robinson fugacities), mole fraction **S**ums and en**H**alpy balances,

$$L_{n-1} x_{j,n-1} + V_{n+1} y_{j,n+1} + F_n z_{j,n} = L_n x_{j,n} + V_n y_{j,n} \qquad \hat{f}^{\,V}_{j,n} = \hat{f}^{\,L}_{j,n}$$

### Column specifications:

| | |
|---|---|
| Feed | 1 kg/s, 50/50 mol benzene/toluene, 300 K, 1.01325 bar, on stage 8 |
| Stages | 13: stage 1 total condenser, stages 2–12 trays, stage 13 reboiler |
| Specifications | reflux ratio 1.5 and bottoms flow 5.8 mol/s |
| Pressure | 1.01325 bar at the condenser, 10 Pa pressure drop to the reboiler |

DWSIM counts the condenser and reboiler as stages, the IDAES `TrayColumn` counts only the trays between them: **IDAES tray k is DWSIM stage k + 1**, so the feed stage 8 is `feed_tray_location=7`.

---

> **Solving strategy**
>
> `distillation_column.py` reaches the solution in four steps:
> 1. initialize tray by tray with the feed near its bubble point (365 K), and a boilup ratio guess;
> 2. tighten the smoothing parameters of the Peng-Robinson vapor-liquid complementarity (`eps_t`, `eps_z`) from their default of 1 mol/s to 10⁻⁴ mol/s. With the default, a phase that should vanish lingers; for example the total condenser outlet at its bubble point keeps about 10 % vapor;
> 3. step the feed temperature down to the actual subcooled 300 K;
> 4. replace the boilup ratio with the DWSIM specification, the bottoms flow.

> **Shortcut vs rigorous**
>
> The shortcut column predicted 12.6 stages for 99 mol % benzene in the distillate and 1 mol % in the bottoms. With 13 stages the rigorous column only reaches 93 % and 5.9 %. The shortcut evaluated the relative volatility at the subcooled feed (α ≈ 3.3), well above its value inside the column (α ≈ 2.4), so it underestimated the stages needed. This is why shortcut designs are checked with a rigorous column.

In [ ]:
import idaes

print("IDAES version: ", idaes.__version__)

## Setting up and solving the flowsheet

The flowsheet is Feed → TrayColumn → distillate and bottoms Products, built in `distillation_column.py`.

In [ ]:
from distillation_column import setup_distillation_model, report_distillation_properties

# Create and solve the model
model, results = setup_distillation_model(
    flow_mass=1.0,  # kg/s
    mole_frac={"benzene": 0.5, "toluene": 0.5},
    temperature=300,  # K
    pressure=101325,  # Pa
    number_of_stages=13,  # DWSIM numbering
    feed_stage=8,  # DWSIM numbering
    reflux_ratio=1.5,
    bottoms_flow=5.8,  # mol/s
    condenser_pressure=101325,  # Pa
    column_pressure_drop=10,  # Pa
)
print(results.solver.termination_condition)

prop = report_distillation_properties(model)

In [ ]:
# Visualize (if needed)
model.fs.visualize("Distillation Column Model")

## Validation against DWSIM

The DWSIM results and converged stage profiles are read directly from the solved `.dwxmz` file.

In [ ]:
from distillation_column import read_dwsim_results, compare_with_dwsim, compare_profiles

dwsim, dwsim_stages = read_dwsim_results("DWSim Distillation Column Model.dwxmz")
compare_with_dwsim(model, dwsim)

In [ ]:
compare_profiles(model, dwsim_stages)

## Stage profiles

IDAES results are shown as lines and DWSIM results as markers. Stage 1 is the condenser and stage 13 the reboiler.

In [ ]:
import matplotlib.pyplot as plt
from distillation_column import profiles

prof = profiles(model)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.plot(prof["Temperature (K)"], prof.index, color="tab:red", label="IDAES")
ax1.plot(dwsim_stages["Temperature (K)"], dwsim_stages.index, "o", color="tab:red", mfc="none", label="DWSIM")
ax1.set_xlabel("Temperature (K)")
ax1.set_ylabel("Stage")
ax1.set_title("Temperature profile")
ax1.invert_yaxis()
ax1.legend()
ax1.grid(alpha=0.3)
ax2.plot(prof["x benzene"], prof.index, color="tab:blue", label="x benzene (IDAES)")
ax2.plot(prof["y benzene"], prof.index, color="tab:orange", label="y benzene (IDAES)")
ax2.plot(dwsim_stages["x benzene"], dwsim_stages.index, "o", color="tab:blue", mfc="none", label="x benzene (DWSIM)")
ax2.plot(dwsim_stages["y benzene"], dwsim_stages.index, "o", color="tab:orange", mfc="none", label="y benzene (DWSIM)")
ax2.set_xlabel("Benzene mole fraction")
ax2.set_title("Composition profile")
ax2.invert_yaxis()
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()